In [159]:
import os, tqdm
import numpy as np
import pandas as pd

In [161]:
# IMDB
imdb_df = pd.read_csv('imdb_crop/imdb.csv')
imdb_df.rename(columns={'paths': 'path'}, inplace=True)
imdb_df['path'] = ['imdb_crop/' + x for x in imdb_df['path']]
imdb_df = imdb_df[['path', 'name']]
imdb_df.head()

,path,name
0,imdb_crop/01/nm0000001_rm124825600_1899-5-10_1...,Fred Astaire
1,imdb_crop/01/nm0000001_rm3343756032_1899-5-10_...,Fred Astaire
2,imdb_crop/01/nm0000001_rm577153792_1899-5-10_1...,Fred Astaire
3,imdb_crop/01/nm0000001_rm946909184_1899-5-10_1...,Fred Astaire
4,imdb_crop/01/nm0000001_rm980463616_1899-5-10_1...,Fred Astaire


In [162]:
name_counts_imdb = imdb_df[['path', 'name']].groupby('name', as_index=False).count()
name_counts_imdb.describe()

,path
count,20284.000000
mean,22.713617
std,56.158003
min,1.000000
25%,2.000000
50%,5.000000
75%,16.000000
max,827.000000


In [179]:
# WIKI
wiki_df = pd.read_csv('wiki_crop/wiki.csv')
wiki_df.rename(columns={'paths': 'path'}, inplace=True)
wiki_df['path'] = ['wiki_crop/' + x for x in wiki_df['path']]
wiki_df = wiki_df[['path', 'name']]
wiki_df.head()

,path,name
0,wiki_crop/17/10000217_1981-05-05_2009.jpg,Sami Jauhojärvi
1,wiki_crop/48/10000548_1925-04-04_1964.jpg,Dettmar Cramer
2,wiki_crop/12/100012_1948-07-03_2008.jpg,Marc Okrand
3,wiki_crop/65/10001965_1930-05-23_1961.jpg,Aleksandar Matanović
4,wiki_crop/16/10002116_1971-05-31_2012.jpg,Diana Damrau


In [181]:
name_counts_wiki = wiki_df[['path', 'name']].groupby('name', as_index=False).count()
name_counts_wiki.describe()

,path
count,61330.000000
mean,1.016273
std,0.515471
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,124.000000


# Dataset splitting into train & val sets

In [329]:
# WIKI
wiki_df_ = wiki_df.drop_duplicates('name')
shuffled_subset = wiki_df_.sample(frac=0.2, random_state=42)
wiki_df['split'] = wiki_df['name'].apply(lambda x: 'val' if x in shuffled_subset['name'].values else 'train')
wiki_df['dataset'] = 'wiki'
# wiki_df.to_csv('wiki.csv', index=False)
wiki_df.groupby('split').count()

,path,name,dataset
split,,,
train,49896,49896,49896
val,12432,12432,12432


In [327]:
# IMDB
imdb_df_ = imdb_df.drop_duplicates('name')
shuffled_subset = imdb_df_.sample(frac=0.2, random_state=42)
imdb_df['split'] = imdb_df['name'].apply(lambda x: 'val' if x in shuffled_subset['name'].values else 'train')
imdb_df['dataset'] = 'imdb'
# imdb_df.to_csv('imdb.csv', index=False)
imdb_df.groupby('split').count()

,path,name,dataset
split,,,
train,371077,371077,371077
val,89646,89646,89646


# lfw-deepfunneled will be used for evaluation purposes only 

In [170]:
os.listdir('lfw-deepfunneled/')

['lfw-deepfunneled',
 'lfw.csv',
 'lfw_allnames.csv',
 'lfw_readme.csv',
 'LICENSE',
 'matchpairsDevTest.csv',
 'matchpairsDevTrain.csv',
 'mismatchpairsDevTest.csv',
 'mismatchpairsDevTrain.csv',
 'pairs.csv',
 'people.csv',
 'peopleDevTest.csv',
 'peopleDevTrain.csv',
 'README.md']

In [120]:
root = 'lfw-deepfunneled/lfw-deepfunneled/'
names = os.listdir(root)

In [147]:
name_list, paths = [], []
for name in names:
    file_names = os.listdir(root + name)
    for file_name in file_names:
        paths.append(root + file_name)
        name_list.append(name)

In [342]:
lfw_df = pd.DataFrame({
    'path': paths,
    'name': name_list,
    'split': 'test',
    'dataset': 'lfw-deepfunneled'
})
# lfw_df.to_csv('lfw-deepfunneled/lfw.csv', index=False)
lfw_df.head()

,path,name,split,dataset
0,lfw-deepfunneled/lfw-deepfunneled/Aaron_Eckhar...,Aaron_Eckhart,test,lfw-deepfunneled
1,lfw-deepfunneled/lfw-deepfunneled/Aaron_Guiel_...,Aaron_Guiel,test,lfw-deepfunneled
2,lfw-deepfunneled/lfw-deepfunneled/Aaron_Patter...,Aaron_Patterson,test,lfw-deepfunneled
3,lfw-deepfunneled/lfw-deepfunneled/Aaron_Peirso...,Aaron_Peirsol,test,lfw-deepfunneled
4,lfw-deepfunneled/lfw-deepfunneled/Aaron_Peirso...,Aaron_Peirsol,test,lfw-deepfunneled


In [103]:
pd.read_csv('lfw-deepfunneled/people.csv').describe()

,images
count,5749.000000
mean,2.301792
std,9.016410
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,530.000000


In [365]:
df = pd.concat([wiki_df, imdb_df, lfw_df]).reset_index(drop=True)
# df.to_csv('wiki_imdb_lfw.csv', index=False)

In [367]:
df.groupby('split').count()

,path,name,dataset
split,,,
test,13233,13233,13233
train,420973,420973,420973
val,102078,102078,102078
